In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

In [3]:
event_logs = pd.read_csv('../filtered_datasets/event_logs(1).csv')
marketing_summary = pd.read_csv('../filtered_datasets/marketing_summary(1).csv')
trend_report = pd.read_csv('../filtered_datasets/trend_report(1).csv')


# 1. Customer Types

In [4]:
event_logs.head()


,user_id,event_type,event_time,product_id,amount
0,U0099,checkout,2023-06-03 04:13:00,P010,NaN
1,U0240,wishlist_add,2023-06-03 05:08:00,P020,0.0
2,U0374,profile_update,2023-06-05 06:22:00,P028,0.0
3,U0122,page_view,2023-06-06 03:45:00,P001,0.0
4,U0211,wishlist_add,2023-06-03 12:38:00,P015,0.0


In [5]:
purchase_counts = (
    event_logs[event_logs['event_type'] == 'checkout']
    .groupby('user_id')
    .size()
    .reset_index(name='purchase_count')
)
user_segments = (
    event_logs[['user_id']]
    .drop_duplicates()
    .merge(purchase_counts, on='user_id', how='left')
)
#replace missing purchases with 0
user_segments['purchase_count'] = (
    user_segments['purchase_count']
    .fillna(0)
    .astype(int)
)

#create segments
user_segments['segment'] = user_segments['purchase_count'].apply(
    lambda x:
        'Non Buyer' if x == 0 else
        'Single Buyer' if x == 1 else
        'Repeat Buyer'
)
user_segments.to_csv('user_segments.csv', index = False)

# 2. Event Logs Forecast

In [7]:
# ============================================================
# Per-product HOURLY sales forecast (checkout amount only)
# Hourly grain kept because daily grain has only 6 data points
# per product (June 1-6), which fails adfuller's minimum sample
# size requirement. Hourly gives ~144 points per product, enough
# to fit ARIMA/SARIMA. Metric changed from total_events/unique_users
# to actual checkout 'amount', since this chart answers a sales
# question, not a generic engagement question.
# ============================================================

checkout_events = event_logs[event_logs['event_type'] == 'checkout'].copy()
checkout_events['event_time'] = pd.to_datetime(checkout_events['event_time'])
checkout_events['event_hour'] = checkout_events['event_time'].dt.floor('h')

# Aggregate sales by hour and product
hourly_sales = (
    checkout_events
    .groupby(['event_hour', 'product_id'])['amount']
    .sum()
    .reset_index(name='sales')
)

forecasts_list = []

for product_id, group in hourly_sales.groupby('product_id'):
    hourly = group.set_index('event_hour').sort_index().asfreq('h').fillna(0)

    p_value = adfuller(hourly['sales'].dropna())[1]

    if p_value >= 0.05:
        model = SARIMAX(hourly['sales'], order=(1,1,1), seasonal_order=(1,0,1,24))
        print(f"{product_id} → SARIMA")
    else:
        model = ARIMA(hourly['sales'], order=(1,0,1))
        print(f"{product_id} → ARIMA")

    result = model.fit()
    forecast_values = result.forecast(steps=168).round().clip(lower=0)  # next 7 days, hourly

    product_forecast = pd.DataFrame({
        'event_hour': forecast_values.index,
        'sales': forecast_values.values,
        'product_id': product_id
    })
    forecasts_list.append(product_forecast)

forecast_df = pd.concat(forecasts_list, ignore_index=True)
forecast_df['type'] = 'forecast'

actual_df = hourly_sales.copy()
actual_df['type'] = 'actual'

combined_sales = pd.concat([actual_df, forecast_df], ignore_index=True)

# add date and week for Tableau filters (forecast horizon: day vs week)
combined_sales['date'] = pd.to_datetime(combined_sales['event_hour']).dt.date
combined_sales['week'] = pd.to_datetime(combined_sales['event_hour']).dt.strftime('%Y-W%V')

combined_sales.to_csv('event_logs_forecast.csv', index=False)
print(combined_sales.head())

P001 → ARIMA
P002 → ARIMA
P003 → ARIMA


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameter

P004 → SARIMA
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  7.27728D+00    |proj g|=  1.06049D-01

At iterate    5    f=  7.25488D+00    |proj g|=  1.49374D-03

At iterate   10    f=  7.25468D+00    |proj g|=  1.94966D-03

At iterate   15    f=  7.25039D+00    |proj g|=  1.51675D-02

At iterate   20    f=  7.24810D+00    |proj g|=  6.96364D-05

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
    5     22     25      1     0     0   7.373D-06   7.248D+00
  F =   7.2481037029900000

 This problem is unconstrained.
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


P005 → ARIMA
P006 → ARIMA


...

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameter

P023 → ARIMA
P024 → ARIMA
P025 → ARIMA
P026 → ARIMA
P027 → ARIMA
P028 → ARIMA
P029 → ARIMA


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameter

P030 → ARIMA
           event_hour product_id    sales    type        date      week
0 2023-06-01 08:00:00       P002     0.00  actual  2023-06-01  2023-W22
1 2023-06-01 09:00:00       P023     0.00  actual  2023-06-01  2023-W22
2 2023-06-01 10:00:00       P025  1708.83  actual  2023-06-01  2023-W22
3 2023-06-01 11:00:00       P003     0.00  actual  2023-06-01  2023-W22
4 2023-06-01 11:00:00       P014  2246.64  actual  2023-06-01  2023-W22


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


# 3. User Intensity


In [8]:
# ============================================================
# user_intensity.csv — overall hourly engagement, ACTUAL ONLY.
# This replaces the old event_logs_forecast.csv approach: same
# aggregation logic, but with the ARIMA/SARIMA forecast step
# removed entirely. No 'type' column needed since there's no
# actual/forecast split — every row here is real.
# ============================================================

event_logs['event_time'] = pd.to_datetime(event_logs['event_time'])
event_logs['event_hour'] = event_logs['event_time'].dt.floor('h')
event_logs['date'] = event_logs['event_time'].dt.date

events_per_hour = (
    event_logs
    .groupby('event_hour')
    .size()
    .reset_index(name='total_events')
)

users_per_hour = (
    event_logs
    .groupby('event_hour')['user_id']
    .nunique()
    .reset_index(name='unique_users')
)

user_intensity = events_per_hour.merge(
    users_per_hour,
    on='event_hour',
    how='outer'
).sort_values('event_hour')

# add date and week for Tableau filters
user_intensity['date'] = pd.to_datetime(user_intensity['event_hour']).dt.date
user_intensity['week'] = pd.to_datetime(user_intensity['event_hour']).dt.strftime('%Y-W%V')

user_intensity.to_csv('user_intensity.csv', index=False)
print(user_intensity.head())

           event_hour  total_events  unique_users        date      week
0 2023-06-01 08:00:00            16            16  2023-06-01  2023-W22
1 2023-06-01 09:00:00            10            10  2023-06-01  2023-W22
2 2023-06-01 10:00:00             5             5  2023-06-01  2023-W22
3 2023-06-01 11:00:00            17            17  2023-06-01  2023-W22
4 2023-06-01 12:00:00            17            17  2023-06-01  2023-W22


# 4. Trend Report Forecast


In [11]:
# ============================================================
# trend_report sales_growth_rate forecast — WEEKLY grain.
# trend_report is inherently weekly (20 weeks of data), so the
# forecast stays at weekly resolution. Any "day vs week" display
# toggle in Tableau is handled via Tableau's own date hierarchy
# (right-click date field -> date part/level), not by fabricating
# repeated daily rows here — keeps the CSV honest to the data's
# real resolution.
# ============================================================
trend_report['week_start'] = pd.to_datetime(trend_report['week_start'])
trend = trend_report.sort_values('week_start').set_index('week_start')
trend = trend.asfreq('W-MON').fillna(0)  # anchor to Monday — matches actual week_start values

p_value = adfuller(trend['sales_growth_rate'].dropna())[1]

if p_value >= 0.05:
    model = SARIMAX(trend['sales_growth_rate'], order=(1,1,1), seasonal_order=(1,0,1,4))
    print(f"sales_growth_rate → SARIMA")
else:
    model = ARIMA(trend['sales_growth_rate'], order=(1,0,1))
    print(f"sales_growth_rate → ARIMA")

result = model.fit()
forecast_values = result.forecast(steps=8).round(4)

forecast_df = pd.DataFrame({
    'week_start': forecast_values.index,
    'sales_growth_rate': forecast_values.values,
    'type': 'forecast'
})

actual_df = trend_report[['week_start', 'sales_growth_rate']].copy()
actual_df['type'] = 'actual'

combined_trend = pd.concat([actual_df, forecast_df], ignore_index=True)
combined_trend['week'] = pd.to_datetime(combined_trend['week_start']).dt.strftime('%Y-W%V')

combined_trend.to_csv('trend_report_forecast.csv', index=False)
print(combined_trend.tail(12))

sales_growth_rate → SARIMA
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f= -1.65961D+00    |proj g|=  2.06354D+00

At iterate    5    f= -1.66204D+00    |proj g|=  7.43667D-01

At iterate   10    f= -1.69857D+00    |proj g|=  1.02011D+01

At iterate   15    f= -1.73128D+00    |proj g|=  1.41813D-01

At iterate   20    f= -1.73518D+00    |proj g|=  2.69390D+00

At iterate   25    f= -1.75681D+00    |proj g|=  2.29316D-01

At iterate   30    f= -1.76207D+00    |proj g|=  1.31067D+00

At iterate   35    f= -1.76864D+00    |proj g|=  4.34546D-01

At iterate   40    f= -1.76910D+00    |proj g|=  1.52149D-01

At iterate   45    f= -1.76913D+00    |proj g|=  5.20169D-02

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS

 This problem is unconstrained.

   evaluations in the last line search.  Termination
   may possibly be caused by a bad search direction.


In [10]:
print(trend_report[['week_start', 'sales_growth_rate']])
print(trend_report['week_start'].diff().value_counts())  # check actual day-gaps between weeks

   week_start  sales_growth_rate
0  2023-05-22             -0.003
1  2023-05-29              0.088
2  2023-06-05              0.073
3  2023-06-12              0.077
4  2023-06-19             -0.003
5  2023-06-26              0.080
6  2023-07-03             -0.015
7  2023-07-10              0.058
8  2023-07-17              0.002
9  2023-07-24              0.095
10 2023-07-31             -0.030
11 2023-08-07              0.050
12 2023-08-14              0.027
13 2023-08-21              0.020
14 2023-08-28              0.099
15 2023-09-04              0.087
16 2023-09-11              0.062
17 2023-09-18              0.007
18 2023-09-25              0.077
19 2023-10-02              0.082
week_start
7 days    19
Name: count, dtype: int64


# 5. Marketing Summary
 

In [12]:
#create key columns for visualization metrics
marketing_summary['date'] = pd.to_datetime(marketing_summary['date'])
marketing_summary = marketing_summary.sort_values('date').set_index('date')
marketing_summary = marketing_summary.asfreq('D').fillna(0)

#forecast them both for 2 forecasts (see canva link for reference if you're making the Tableau charts)

marketing_forecasts = {}

for col in ['new_customers', 'total_sales']:
    p_value = adfuller(marketing_summary[col].dropna())[1]
    
    if p_value >= 0.05:
        model = SARIMAX(marketing_summary[col], order=(1,1,1), seasonal_order=(1,0,1,7))
        print(f"{col} → SARIMA")
    else:
        model = ARIMA(marketing_summary[col], order=(1,0,1))
        print(f"{col} → ARIMA")
    
    result = model.fit()
    marketing_forecasts[col] = result.forecast(steps=7).round().clip(lower=0)

print(marketing_forecasts)
#label and export

actual_df = marketing_summary[['new_customers', 'total_sales']].copy()
actual_df['type'] = 'actual'
actual_df = actual_df.reset_index()  # brings date back as column

forecast_df = pd.DataFrame({
    'date': marketing_forecasts['new_customers'].index,
    'new_customers': marketing_forecasts['new_customers'].values,
    'total_sales': marketing_forecasts['total_sales'].values,
    'type': 'forecast'
})

combined_marketing = pd.concat([actual_df, forecast_df], ignore_index=True)

# add week column for Tableau filter
combined_marketing['week'] = pd.to_datetime(combined_marketing['date']).dt.strftime('%Y-W%V')

combined_marketing.to_csv('marketing_summary_forecast.csv', index=False)
print(combined_marketing)

new_customers → ARIMA
total_sales → ARIMA
{'new_customers': 2023-09-09    8.0
2023-09-10    8.0
2023-09-11    8.0
2023-09-12    8.0
2023-09-13    8.0
2023-09-14    8.0
2023-09-15    8.0
Freq: D, Name: predicted_mean, dtype: float64, 'total_sales': 2023-09-09    60347.0
2023-09-10    58721.0
2023-09-11    58085.0
2023-09-12    57836.0
2023-09-13    57739.0
2023-09-14    57700.0
2023-09-15    57685.0
Freq: D, Name: predicted_mean, dtype: float64}
          date  new_customers  total_sales      type      week
0   2023-06-01            9.0     81287.31    actual  2023-W22
1   2023-06-02            5.0     74771.99    actual  2023-W22
2   2023-06-03           11.0     84809.74    actual  2023-W22
3   2023-06-04            3.0     61212.30    actual  2023-W22
4   2023-06-05           10.0     80911.49    actual  2023-W23
..         ...            ...          ...       ...       ...
102 2023-09-11            8.0     58085.00  forecast  2023-W37
103 2023-09-12            8.0     57836.00  for